I will be creating a model to predict whether a patient that is in the ICU will develop sepsis or not using the dataset from Physionet which gave over 40,000 total patients and whether they developed sepsis. Each patient has one row per hour that they stayed in the ICU; the goal of each row is to show their vitals/labs at that hour.

In [77]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

Now I will be reading in the data that I downloaded into two PSV files, there was training set A and training set B

In [13]:
import glob #grabs each file path that matches a pattern
import os #gets the patient id from each file name we are at

patientsA = [] #stores each patient df
for patient in glob.glob(r'C://training//training_setA/*.psv'):
    df = pd.read_csv(patient, delimiter='|') #stores current patient as a df
    df['patient_id'] = os.path.basename(patient)[:-4] #saves the id part from the path for the patient id column
    patientsA.append(df) #store in the array
dfA = pd.concat(patientsA, ignore_index=True) #creates our set A df using all of the patients

#now for set B
patientsB = []
for patient in glob.glob(r'C://training//training_setB/*.psv'):
    df = pd.read_csv(patient, delimiter='|')
    df['patient_id'] = os.path.basename(patient)[:-4]
    patientsB.append(df)
dfB = pd.concat(patientsB, ignore_index=True)

Now combining each set into one dataframe and then saving it so we don't need to load the data each time

In [28]:
df = pd.concat([dfA,dfB], ignore_index=True)
df.to_pickle('sepsis_df')

Now going to get some of the stats about the dataset

In [116]:
df = pd.read_pickle('sepsis_df')
shape = df.shape #shape of the data
sepsis_percent = len(df[df['SepsisLabel']==1]) / len(df) #percentage of rows that actually had sepsis
#info = df.info()
#describe = df.describe()
num_patients = len(df.groupby('patient_id')) #there are 40,336 patients here
stay_len = df.groupby('patient_id')['ICULOS'].max() #gets the last hour of each patients ICU stay-- that col is holding the current number of hours the patient has been in the ICU
stay_len.describe() #shows the stats about the general patient length of stay in the ICU in hours

count    40336.000000
mean        39.010115
std         22.924641
min          8.000000
25%         24.000000
50%         39.000000
75%         47.000000
max        336.000000
Name: ICULOS, dtype: float64

Now splitting the data by patient (keeping whole patient records together) and stratify on the proporiton of patients that have had sepsis

In [136]:
patients = df.groupby('patient_id')['SepsisLabel'].max() #each patient will hold max of SepsisLabel, since its binary it will either by 1 if they got it or 0 if not
patients_train, patients_test = train_test_split(patients, test_size=0.2, stratify=patients, random_state=445)
#checking to see if the sepsis positives were distributed evenly
perc1 = sum(patients_train==1) / len(patients_train)
perc2 = sum(patients_test==1) / len(patients_test)

Building the dataframes back up using the patients that fell into both

In [143]:
train_df = df[df['patient_id'].isin(patients_train.index)]
test_df = df[df['patient_id'].isin(patients_test.index)]
p1 = sum(train_df['SepsisLabel']==1) / len(train_df)
p2 = sum(test_df['SepsisLabel']==1) / len(test_df)
p1

0.0180131431350143